# 06 · Final pipeline: detection budget and ensembling

Part of *Your pipeline*. Two validated detectors of similar accuracy but different families:
`yolo_1024` (anchor-free CNN, 1024 px, OOF mAP 0.189) and `rtdetr_stable` (query-based, 640 px,
OOF mAP 0.183). Their errors differ (RT-DETR has the higher recall), so combining them is promising.

Steps, all on the **fold models** (each trained with the validated recipe on 2/3 of the clips):
1. **Detection budget:** the metric scores up to 500 boxes per image (VisDrone protocol); the
   comparison runs returned 300. `max_det` 300 vs 500 is measured on the same predictions.
2. **Ensemble:** Weighted Boxes Fusion of YOLO and RT-DETR (`ensemble.py`), weight and IoU
   threshold chosen leave-one-fold-out.
3. **Test submissions** from the 3 + 3 fold models: YOLO folds only (checks whether the full-data
   final model overfitted: it scored 0.219 public vs 0.239 for the 640 px model) and YOLO + RT-DETR.

Modes: `predict` (GPU; attach the outputs that contain `ul_runs/yolo_1024` and `ul_runs/rtdetr_stable`)
and `tune` (CPU is enough; attach also the output of the `predict` run).

## 1. Run configuration

In [ ]:
MODE = "tune"              # "predict" (GPU) | "tune" (CPU)
MODELS = {                 # experiment -> inference settings of its fold models
    "yolo_1024": dict(model="yolo", imgsz=1024),
    "rtdetr_stable": dict(model="rtdetr", imgsz=640),
}
MAX_DET = 500
QUICK = False              # True: few frames (code check only)

## 2. Setup

In [ ]:
import os, sys, shutil, subprocess, time, urllib.request
from pathlib import Path

os.environ["YOLO_VERBOSE"] = "False"
try:
    import ultralytics, ensemble_boxes
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--timeout", "60", "--retries", "5",
                    "ultralytics==8.4.203", "pycocotools", "ensemble-boxes"], check=True, timeout=1800)
os.environ["WANDB_MODE"] = "disabled"

REPO_RAW = "https://raw.githubusercontent.com/KaplyshOlha/hw-2/main/notebooks/"

def fetch(url, dst, timeout=60, tries=5):
    # download with a timeout and retries: a stalled GitHub request must not hang a commit for hours
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as r:
                Path(dst).write_bytes(r.read())
            return dst
        except Exception as e:
            if i == tries - 1:
                raise
            print(f"{url}: {type(e).__name__}, retry in 30 s"); time.sleep(30)

SRC_FILES = ["common.py", "ul_dataset.py", "ensemble.py", "splits.json"]
SRC = next((Path(p) for p in [".", "notebooks"] if all((Path(p) / f).exists() for f in SRC_FILES)), None)
if SRC is None:
    # GitHub is the source of truth; an attached copy is only a fallback when GitHub is unreachable
    SRC = Path("/tmp/hw2_src"); SRC.mkdir(exist_ok=True)
    for f in SRC_FILES:
        try:
            fetch(REPO_RAW + f, SRC / f, tries=3); print(f"{f}: GitHub")
        except Exception:
            found = [p for pat in ["*", "*/*", "*/*/*"] for p in sorted(Path("/kaggle/input").glob(f"{pat}/{f}"))
                     if "competitions" not in p.parts] if Path("/kaggle/input").exists() else []
            if not found:
                raise
            shutil.copy(found[0], SRC / f); print(f"{f}: {found[0]} (GitHub unreachable)")
sys.path.insert(0, str(SRC))

In [ ]:
import importlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

import common, ul_dataset as uld, ensemble as E
importlib.reload(common); importlib.reload(uld); importlib.reload(E)

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
RUN_ROOT = WORK / "ul_runs"
PRED_DIR = WORK / "ens_preds"
IMG_DIR = WORK / "images"
RESULTS = WORK / ("results.csv" if not QUICK else "results_quick.csv")
for p in (RUN_ROOT, PRED_DIR, IMG_DIR):
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "0" if torch.cuda.is_available() else "cpu"
if MODE == "predict" and not QUICK:
    assert torch.cuda.is_available(), "attach a GPU (Settings -> Accelerator)"

images, anns = common.load_gt(cache="/tmp/gt_cache.pkl")
splits = common.load_splits(SRC / "splits.json")
test = common.load_test_index()
if QUICK:
    test = test.groupby("clip").head(3).head(12)
print("device:", DEVICE, "| mode:", MODE)

## 3. Predictions (`MODE = "predict"`)
Every fold model predicts its fold's protocol validation frames (OOF) and all test frames,
with `conf = 0.001` and `max_det = 500`.

In [ ]:
if MODE == "predict":
    uld.restore_from_inputs(RUN_ROOT)
    for exp, s in MODELS.items():
        cfg = uld.Config(experiment=exp, model=s["model"], imgsz=s["imgsz"], max_det=MAX_DET,
                         device=DEVICE, pred_batch=16 if s["model"] == "yolo" else 8)
        for k in range(common.N_FOLDS):
            w = RUN_ROOT / exp / f"fold{k}" / "final.pt"
            assert w.exists(), f"{w} missing: attach the notebook output that trained {exp}"
            model = uld.load_trained(cfg, w)
            t0 = time.time()
            ev = common.fold_images(images, splits, k, "val", common.EVAL_STRIDE)
            if QUICK:
                ev = ev.head(6)
            p = uld.predict(model, ev["path"].tolist(), ev["img_id"].tolist(), cfg)
            p.rename(columns={"key": "img_id"}).to_parquet(PRED_DIR / f"{exp}_fold{k}_oof.parquet")
            pt = uld.predict(model, test["path"].tolist(), test["image_id"].tolist(), cfg)
            pt.rename(columns={"key": "image_id"}).to_parquet(PRED_DIR / f"{exp}_fold{k}_test.parquet")
            print(f"{exp} fold {k}: {len(ev)} OOF frames, {len(test)} test frames, {(time.time() - t0) / 60:.1f} min")

## 4. Load predictions (`MODE = "tune"`)

In [ ]:
def load_pred(name):
    f = PRED_DIR / name
    hit = [f] if f.exists() else uld._input_files("/kaggle/input", f"ens_preds/{name}")
    assert hit, f"{name} missing: attach the output of the predict run"
    return pd.read_parquet(hit[0])

if MODE == "tune":
    oof = {(e, k): load_pred(f"{e}_fold{k}_oof.parquet") for e in MODELS for k in range(common.N_FOLDS)}
    tst = {(e, k): load_pred(f"{e}_fold{k}_test.parquet") for e in MODELS for k in range(common.N_FOLDS)}
    sizes = images.set_index("img_id")[["width", "height"]]
    print({e: sum(len(oof[(e, k)]) for k in range(common.N_FOLDS)) for e in MODELS}, "OOF boxes")

## 5. Detection budget: `max_det` 300 vs 500
The 300-box result is the top-300 subset of the same 500-box predictions, so the difference is
only the extra low-score boxes.

In [ ]:
def oof_metrics(pf):
    return common.evaluate_oof(pf, images, anns, splits, common.EVAL_STRIDE)

MAIN = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR500"]
if MODE == "tune":
    table = {}
    for e in MODELS:
        for kmax in [300, 500]:
            pf = {k: E.topk(oof[(e, k)], kmax, key="img_id") for k in range(common.N_FOLDS)}
            table[f"{e}_max{kmax}"] = oof_metrics(pf)
    display(pd.DataFrame({n: m.loc["oof", MAIN] for n, m in table.items()}).T.round(4))

**Observations (detection budget):** TBD

## 6. WBF ensemble of YOLO and RT-DETR
Grid over the RT-DETR weight (YOLO weight 1) and the WBF IoU threshold; for every fold the setting
with the best mean mAP on the other two folds is used (leave-one-fold-out).

In [ ]:
GRID = [dict(w_rt=w, iou_thr=t) for w in [0.5, 1.0, 2.0] for t in [0.55, 0.65, 0.75]]
if QUICK:
    GRID = GRID[:2]
names = list(MODELS)

def fuse_fold(k, w_rt, iou_thr):
    a = oof[(names[0], k)].rename(columns={"img_id": "key"})
    b = oof[(names[1], k)].rename(columns={"img_id": "key"})
    f = E.fuse([a, b], sizes, weights=[1.0, w_rt], iou_thr=iou_thr, max_det=MAX_DET)
    return f.rename(columns={"key": "img_id"}).astype({"img_id": int})

if MODE == "tune":
    ens = {}
    t0 = time.time()
    for g in GRID:
        n = f"wbf_w{g['w_rt']}_iou{g['iou_thr']}"
        ens[n] = (g, oof_metrics({k: fuse_fold(k, **g) for k in range(common.N_FOLDS)}))
        print(f"{n:22s} OOF mAP {ens[n][1].at['oof', 'mAP']:.4f}  [{(time.time() - t0) / 60:.1f} min]")
    choice = {}
    for k in range(common.N_FOLDS):
        others = [j for j in range(common.N_FOLDS) if j != k]
        choice[k] = max(ens, key=lambda n: np.mean([ens[n][1].at[j, "mAP"] for j in others]))
    print("leave-one-fold-out choice:", choice)
    m_ens = oof_metrics({k: fuse_fold(k, **ens[choice[k]][0]) for k in range(common.N_FOLDS)})
    table["wbf_yolo+rtdetr (LOFO)"] = m_ens

Summary on the same folds and frames (OOF; mean ± std over folds).

In [ ]:
if MODE == "tune":
    summ = pd.DataFrame({n: {**m.loc["oof", MAIN].to_dict(),
                             "mAP mean±std": f"{m.at['mean', 'mAP']:.4f} ± {m.at['std', 'mAP']:.4f}"}
                         for n, m in table.items()}).T
    display(summ)
    fig, ax = plt.subplots(figsize=(8, 3))
    pd.DataFrame({n: m.loc["oof", ["mAP", "AP_small", "AP_medium", "AP_large"]] for n, m in table.items()}).T \
        .plot.barh(ax=ax)
    ax.set_xlabel("OOF AP"); fig.tight_layout(); fig.savefig(IMG_DIR / "06_ensemble.png"); plt.show()

**Observations (ensemble):** TBD

## 7. Logging

In [ ]:
if MODE == "tune":
    logged = WORK / ("logged_quick" if QUICK else "logged")
    if not logged.exists():
        common.log_results(table["yolo_1024_max500"], RESULTS, owner="B", model="yolo", experiment="yolo_1024_max500",
                           train_stride=20, eval_stride=10, imgsz=1024, epochs=12, notes="max_det 500 at inference")
        common.log_results(table["rtdetr_stable_max500"], RESULTS, owner="B", model="rtdetr",
                           experiment="rtdetr_stable_max500", train_stride=20, eval_stride=10, imgsz=640, epochs=12,
                           notes="max_det 500 at inference (decoder returns <= 300)")
        common.log_results(m_ens, RESULTS, owner="B", model="ensemble", experiment="wbf_yolo1024_rtdetr",
                           train_stride=20, eval_stride=10, epochs=12,
                           notes="WBF yolo_1024 + rtdetr_stable, LOFO: " + str({k: v for k, v in choice.items()}))
        logged.touch()
    print(pd.read_csv(RESULTS).query("fold == 'oof'")[["experiment", "mAP", "AP_small"]])

## 8. Test submissions from the fold models
- `submission_yolo_folds.csv`: WBF of the 3 `yolo_1024` fold models (no full-data retraining).
- `submission_ensemble.csv`: WBF of all 6 fold models, YOLO and RT-DETR weighted with the
  setting that is best on all three folds.

In [ ]:
if MODE == "tune":
    tsizes = test.set_index("image_id")[["width", "height"]]
    sample = WORK / "sample_from_index.csv"
    test[["image_id"]].assign(PredictionString="").to_csv(sample, index=False)
    best = max(ens, key=lambda n: ens[n][1].at["mean", "mAP"])
    g = ens[best][0]
    print("ensemble setting for test:", best)
    yolo_t = [tst[(names[0], k)].rename(columns={"image_id": "key"}) for k in range(common.N_FOLDS)]
    rt_t = [tst[(names[1], k)].rename(columns={"image_id": "key"}) for k in range(common.N_FOLDS)]
    t0 = time.time()
    sub_y = E.fuse(yolo_t, tsizes, iou_thr=g["iou_thr"], max_det=MAX_DET)
    sub_e = E.fuse(yolo_t + rt_t, tsizes, weights=[1.0] * 3 + [g["w_rt"]] * 3, iou_thr=g["iou_thr"], max_det=MAX_DET)
    for name, p in [("submission_yolo_folds.csv", sub_y), ("submission_ensemble.csv", sub_e)]:
        out = common.write_submission(p.rename(columns={"key": "image_id"}), test, WORK / name)
        print(name, common.check_submission(out, sample))
    print(f"{(time.time() - t0) / 60:.1f} min")

## 9. Conclusions
TBD